# Etapa 4

In [1]:
# criando seção spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("INPE-MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

d:\data_science\inpe-mllib-study\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
df = spark.read.parquet(
    "../data/silver/"
)

Antes de qualquer coisa do pipeline, vamos fazer uma breve análise das variáveis das nossas hipóteses agora

## Testando hipóteses

### Hipótese principal:

Menor precipitação/maior período sem chuva e menor umidade aumentam a probabilidade de FOCO_INTENSO = 1.

Analisando as possíveis variáveis temos seca_prolongada, baixa_umidade, condicao_seca e podemos criar mais uma

In [17]:
df.groupBy("seca_prolongada").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).show()

+---------------+-------+------------------+
|seca_prolongada|      n| taxa_foco_intenso|
+---------------+-------+------------------+
|              1|1130038|0.4923064534113012|
|              0|3399157| 0.497963171456923|
+---------------+-------+------------------+



In [18]:
df.groupBy("baixa_umidade").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).show()

+-------------+-------+------------------+
|baixa_umidade|      n| taxa_foco_intenso|
+-------------+-------+------------------+
|            1|1215253|0.6065123887783038|
|            0|3313942|0.4562282622930637|
+-------------+-------+------------------+



In [19]:
df.groupBy("condicao_seca").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).show()

+-------------+-------+-------------------+
|condicao_seca|      n|  taxa_foco_intenso|
+-------------+-------+-------------------+
|            1| 406508| 0.5852799944896533|
|            0|4122687|0.48780297897948593|
+-------------+-------+-------------------+



Vamos testar também a precipitação acumulada que acabamos esquecendo, para isso vamos testar uma faixa de precipitação acumulada no dia:

In [20]:
df = df.withColumn(
    "faixa_precipitacao",
    F.when(F.col("precipitacao_inpe_mm") == 0, "0 mm")
     .when(F.col("precipitacao_inpe_mm") <= 1, "0-1 mm")
     .when(F.col("precipitacao_inpe_mm") <= 5, "1-5 mm")
     .otherwise(">5 mm")
)

In [22]:
df.groupBy("faixa_precipitacao").agg(
    F.count("*").alias("n"),
    F.avg("FOCO_INTENSO").alias("taxa_foco_intenso")
).orderBy("faixa_precipitacao").show()

+------------------+-------+------------------+
|faixa_precipitacao|      n| taxa_foco_intenso|
+------------------+-------+------------------+
|              0 mm|3971998|0.5024332842060847|
|            0-1 mm| 313455|0.4586240449187284|
|            1-5 mm| 158826|0.4568521526702177|
|             >5 mm|  84916|0.4357011635027557|
+------------------+-------+------------------+



In [23]:
df.groupBy("FOCO_INTENSO").agg(
    F.count("*").alias("n"),
    
    F.avg("umidade_relativa_ar_pct").alias(
        "umidade_media"
    ),
    
    F.avg("dia_sem_chuva").alias(
        "dias_sem_chuva_medio"
    ),
    
    F.avg("precipitacao_inpe_mm").alias(
        "precipitacao_media"
    )
).show()

+------------+-------+------------------+--------------------+-------------------+
|FOCO_INTENSO|      n|     umidade_media|dias_sem_chuva_medio| precipitacao_media|
+------------+-------+------------------+--------------------+-------------------+
|           1|2248980| 36.46994103993811|   32.93028839740683|0.32000678974469066|
|           0|2280215|41.624881864210174|   33.41522663433054| 0.3995076078352584|
+------------+-------+------------------+--------------------+-------------------+



A análise descritiva inicial apresentou indícios favoráveis à hipótese principal principalmente em relação à precipitação e à umidade. A proporção de focos intensos diminuiu conforme aumentaram os níveis de precipitação, passando de aproximadamente 50,24% nos registros sem precipitação para 43,57% nos registros acima de 5 mm. Da mesma forma, os focos classificados como intensos apresentaram umidade média menor (36,47%) do que os não intensos (41,62%). Por outro lado, dia_sem_chuva não apresentou uma diferença relevante entre as classes, com médias de 32,93 e 33,42 dias. Dessa forma, os resultados iniciais dão suporte principalmente à relação entre baixa precipitação, baixa umidade e focos mais intensos, enquanto o efeito da duração do período sem chuva ainda não apresenta evidências claras.

### Hipótese secundária 1:



4A — Preparação para o MLlib
- Converter colunas categóricas com StringIndexer + OneHotEncoder
- Montar o vetor de features com VectorAssembler
- Dividir os dados em treino (70%) e teste (30%) usando randomSplit()
- Documentar a contagem de registros em cada split

Primeiramente vamos converter as colunas categóricas:

In [4]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

categoricas = [
    "periodo_dia",
    "bioma",
    "estado",
    "satelite"
]

indexers = [
    StringIndexer(
        inputCol=coluna,
        outputCol=f"{coluna}_idx",
        handleInvalid="keep"
    )
    for coluna in categoricas
]

encoders = [
    OneHotEncoder(
        inputCol=f"{coluna}_idx",
        outputCol=f"{coluna}_ohe",
        handleInvalid="keep"
    )
    for coluna in categoricas
]

Agora vamos montar o vetor de features

In [6]:
from pyspark.ml.feature import VectorAssembler

numericas = [
    "dia_sem_chuva",
    "precipitacao_inpe_mm",
    "umidade_relativa_ar_pct",
    "seca_prolongada",
    "baixa_umidade",
    "condicao_seca",
    "mes",
    "distancia_estacao_km",
    "dia_sem_chuva_missing",
    "precipitacao_missing",
    "umidade_missing"
]

assembler = VectorAssembler(
    inputCols=numericas + [
        "periodo_dia_ohe",
        "bioma_ohe",
        "estado_ohe",
        "satelite_ohe"
    ],
    outputCol="features"
)

Agora vamos dividir os dados em treino (70%) e teste (30%) usando randomSplit()

In [7]:
train, test = df.randomSplit(
    [0.7, 0.3],
    seed=42
)

In [14]:
df.count()

4529195

In [12]:
qtd_train = train.count()
qtd_train

3171055

In [11]:
qtd_test = test.count()
qtd_test

1358140

Tivemos 3171055 registros no treino e 1358140 registros no teste, de um total de 4529195 registros

4B — Construção do Pipeline
- Montar um Pipeline encadeando os estágios de preparação e o modelo, na ordem correta:
StringIndexer(s) → OneHotEncoder(s) → VectorAssembler → Modelo
- Treinar pelo menos 2 modelos diferentes do MLlib:
LogisticRegression e RandomForestClassifier

Vamos construir o pipeline agora dos nossos modelos:

In [16]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

lr = LogisticRegression(
    featuresCol="features",
    labelCol="FOCO_INTENSO"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="FOCO_INTENSO",
    seed=42
)

pipeline_lr = Pipeline(
    stages=
        indexers +
        encoders +
        [assembler, lr]
)

pipeline_rf = Pipeline(
    stages=
        indexers +
        encoders +
        [assembler, rf]
)